In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import sys
sys.path.insert(0, r"c:\users\arthu\documents\swittz-epfl\phd\code\lib\site-packages")
sys.path.append('../')
import igraph as ig

In [ ]:
from data_extract import extract_wdpli
base_path_surv = "C:\\Users\\arthu\\Desktop\\Marzia\\marzia_data\\data_anonym_correct\\surv"
base_path_nsurv = "C:\\Users\\arthu\\Desktop\\Marzia\\marzia_data\\data_anonym_correct\\nonsurv"
dataframes_wdpli_surv, folders_surv = extract_wdpli(base_path_surv, surv=True)
dataframes_wdpli_nsurv, folders_nsurv = extract_wdpli(base_path_nsurv, surv=False)

print("Number of subjects in surv: ", len(folders_surv))
print("Number of subjects in non-surv: ", len(folders_nsurv))

In [ ]:
from data_extract import adj_percentile_thresholding
freq_list = np.arange(1,41)
thresholds = np.arange(50, 91, 10)#[50, 65, 80]
adj_mat_percentile_surv = {}
adj_mat_percentile_nsurv = {}
dict_id_pass_surv = []
dict_id_pass_nsurv = []

for p, patient in enumerate(folders_surv):
    for f, freq in enumerate(freq_list):
        for t, tau in enumerate(thresholds):
            data = np.nan_to_num(dataframes_wdpli_surv[f'{patient}_dwPLI_1_40Hz_wpli_debiasedspctrm_slice{f+1}'].values)
            mat = adj_percentile_thresholding(data, tau)
            if mat.sum() == 0:
                dict_id_pass_surv.append((patient, freq, tau))
                print(f"Patient {patient} has a zero matrix at freq {freq} and threshold {tau}")
                pass
            else:
                adj_mat_percentile_surv[(patient, freq, tau)] = mat

for p, patient in enumerate(folders_nsurv):
    for f, freq in enumerate(freq_list):
        for t, tau in enumerate(thresholds):
            data = np.nan_to_num(dataframes_wdpli_nsurv[f'{patient}_dwPLI_1_40Hz_wpli_debiasedspctrm_slice{f+1}'].values)
            mat = adj_percentile_thresholding(data, tau)
            if mat.sum() == 0:
                dict_id_pass_nsurv.append((patient, freq, tau))
                print(f"Patient {patient} has a zero matrix at freq {freq} and threshold {tau}")
                pass
            else:
                adj_mat_percentile_nsurv[(patient, freq, tau)] = mat

In [ ]:
from pygraphon.estimators import USVT, NBDsmooth

estimators_percentile_surv = {}
estimators_percentile_nsurv = {}

for p, patient in enumerate(folders_surv):
    for f, freq in enumerate(freq_list):
        for t, tau in enumerate(thresholds):
            if (patient, freq, tau) in dict_id_pass_surv:
                pass
            else:
                estimator = USVT()
                estimator.fit(adj_mat_percentile_surv[(patient, freq, tau)])
                estimators_percentile_surv[(patient, freq, tau)] = estimator

for p, patient in enumerate(folders_nsurv):
    for f, freq in enumerate(freq_list):
        for t, tau in enumerate(thresholds):
            if (patient, freq, tau) in dict_id_pass_nsurv:
                pass
            else:
                estimator = USVT()
                estimator.fit(adj_mat_percentile_nsurv[(patient, freq, tau)])
                estimators_percentile_nsurv[(patient, freq, tau)] = estimator

In [ ]:
connectivity_surv = {}
connectivity_nsurv = {}

for f, freq in enumerate(freq_list):
    for t, tau in enumerate(thresholds):
        conn_surv = np.zeros((63,63))
        conn_nsurv = np.zeros((63,63))
        for p, patient in enumerate(folders_surv):
            if (patient, freq, tau) in dict_id_pass_surv:
                pass
            else:
                conn_surv += estimators_percentile_surv[(patient, freq, tau)].get_edge_connectivity()
        for p, patient in enumerate(folders_nsurv):
            if (patient, freq, tau) in dict_id_pass_nsurv:
                pass
            else:
                conn_nsurv += estimators_percentile_nsurv[(patient, freq, tau)].get_edge_connectivity()
        connectivity_surv[(freq, tau)] = conn_surv/len(folders_surv)
        connectivity_nsurv[(freq, tau)] = conn_nsurv/len(folders_nsurv)

In [ ]:
from metrics import auc_ppv_score, t_test_topo_features, mannwhitneyu_topo_features
aucs = {}
ppvs = {}
for t, tau in enumerate(thresholds):
    # for each tau, store the (i,j)th element of the .get_edge_connectivity() matrix for the first dimension and second should be frequency
    for i in range(63):
        for j in range(63):
            surv = np.zeros((len(folders_surv), len(freq_list)))
            nsurv = np.zeros((len(folders_nsurv), len(freq_list)))
            for p, patient in enumerate(folders_surv):
                for f, freq in enumerate(freq_list):
                    if (patient, freq, tau) in dict_id_pass_surv:
                        pass
                    else:
                        surv[p, f] = estimators_percentile_surv[(patient, freq, tau)].get_edge_connectivity()[i, j]
            for p, patient in enumerate(folders_nsurv):
                for f, freq in enumerate(freq_list):
                    if (patient, freq, tau) in dict_id_pass_nsurv:
                        pass
                    else:
                        nsurv[p, f] = estimators_percentile_nsurv[(patient, freq, tau)].get_edge_connectivity()[i, j]
            auc, ppv = auc_ppv_score(surv, nsurv)
            for f, freq in enumerate(freq_list):
                aucs[(i+1, j+1, freq, tau)] = auc[f]
                ppvs[(i+1, j+1, freq, tau)] = ppv[f]


In [ ]:
#find in aucs, the top 5 values and their corresponding indices in the dictionnary
top_auc = sorted(aucs.items(), key=lambda x: x[1], reverse=True)[:10]
# print them with their corresponding ppv from ppvs
for i in range(10):
    print(top_auc[i], ppvs[top_auc[i][0]])

In [ ]:
#do the smae but for .get_edge_connectivity() colmuns average
aucs_mean = {}
ppvs_mean = {}
for t, tau in enumerate(thresholds):
    # for each tau, store the (i,j)th element of the .get_edge_connectivity() matrix for the first dimension and second should be frequency
    for i in range(63):
        surv = np.zeros((len(folders_surv), len(freq_list)))
        nsurv = np.zeros((len(folders_nsurv), len(freq_list)))
        for p, patient in enumerate(folders_surv):
            for f, freq in enumerate(freq_list):
                if (patient, freq, tau) in dict_id_pass_surv:
                    pass
                else:
                    surv[p, f] = estimators_percentile_surv[(patient, freq, tau)].get_edge_connectivity()[i, :].mean()
        for p, patient in enumerate(folders_nsurv):
            for f, freq in enumerate(freq_list):
                if (patient, freq, tau) in dict_id_pass_nsurv:
                    pass
                else:
                    nsurv[p, f] = estimators_percentile_nsurv[(patient, freq, tau)].get_edge_connectivity()[i, :].mean()
        auc, ppv = auc_ppv_score(surv, nsurv)
        for f, freq in enumerate(freq_list):
            aucs_mean[(i+1, freq, tau)] = auc[f]
            ppvs_mean[(i+1, freq, tau)] = ppv[f]

In [ ]:
top_auc_mean = sorted(aucs_mean.items(), key=lambda x: x[1], reverse=True)[:10]
for i in range(10):
    print(top_auc_mean[i], ppvs_mean[top_auc_mean[i][0]])